#📚 文系大学生による『実践データ分析100本ノック＆LLM活用20本ノック』学習ログ
**著： 岡野 菜月（四国大学文学部 / 28卒）**

**GitHub： https://github.com/NatsukiOkano**

##📝 このノートブックの概要
＊ 書籍「Python 実践データ分析100本ノック 第3版」（秀和システム新社刊）の学習過程を記録したものです。

＊ 最大の特徴は、「文学部の視点でIT・データ分析を再解釈する」という試みにあります。

＊ 非情報系の学習者が直面する「専門用語の壁」を取り払うため、すべての処理に対し、論理的かつ直感的な「完全独自解説」を記述しています。

##⚠️ 注意事項
- **著作権保護の観点から、データセットおよび実行結果は同梱しておりません。**

### 教材
下山輝昌・松田雄馬・三木孝行 著「Python 実践データ分析100本ノック 第3版」（秀和システム新社、2025）

# １章 ウェブの注文数を分析する１０本ノック

ここでは、ある企業のECサイトでの商品の注文数の推移を分析していきます。  
データの属性を理解し、分析をするためにデータを加工した後、  
データの可視化を行うことで問題を発見していくプロセスを学びます。

### ノック１：データを読み込んでみよう

In [ ]:
import pandas as pd
customer_master = pd.read_csv('customer_master.csv')
customer_master.head()

#import：外にある便利な「道具箱」を、自分の手元に用意する。
#as：そのままだと名前が長くて使いにくいため、短い「あだ名」を付けて、使いやすくする。
#pd.read_csv()：CSVファイルの読み込み

In [ ]:
item_master = pd.read_csv('item_master.csv')
item_master.head()

In [ ]:
transaction_1 = pd.read_csv('transaction_1.csv')
transaction_1.head()

#.head()：データの最初の5行を表示

In [ ]:
transaction_2 = pd.read_csv('transaction_2.csv')
transaction_2.head()

In [ ]:
transaction_detail_1 = pd.read_csv('transaction_detail_1.csv')
transaction_detail_1.head()

In [ ]:
transaction_detail_2 = pd.read_csv('transaction_detail_2.csv')
transaction_detail_2.head()

### ノック２：データを結合(ユニオン)してみよう

In [ ]:
transaction = pd.concat([transaction_1,transaction_2],ignore_index=True)
transaction.head()

#pd.concat()：結合。同じ形式(列名)のデータフレーム(表)を、縦(行方向)に結合する。
#ignore_index：0から順番に行番号(index)を新しく振り直す。
#ignore_index=True：0から順番に行番号(index)を新しく振り直す。
#ignore_index=False：0から順番に行番号(index)を新しく振り直さず、元の行番号(index)を使う。

In [ ]:
transaction_detail = pd.concat([transaction_detail_1,transaction_detail_2],ignore_index=True)
transaction_detail.head()

### ノック３：売上データ同士を結合(ジョイン)してみよう

In [ ]:
join_data = pd.merge(transaction_detail, transaction[['transaction_id','payment_date', 'customer_id']], on='transaction_id', how='left')

#pd.merge()：結合。正規化されてバラバラになったデータを、分析しやすいように「逆正規化（合体）」させる。
#join_data：変数。

In [ ]:
join_data.head()

In [ ]:
print(len(transaction_detail))
print(len(transaction))
print(len(join_data))

#print：目に見えるように、外に出して表示すること。
#len：データの件数を数える。Excelのcount関数と同じ役割。

### ノック４：マスタデータを結合(ジョイン)してみよう

In [ ]:
join_data = pd.merge(join_data, customer_master, on='customer_id', how='left')

#on：「どの列を基準に結合するか」を指定。
#how：「どのような結合方法を用いるか」を指定。（結合方法↓）
#結合方法'inner'：デフォルト。両方のデータフレームにキーが存在する行のみを残す。
#結合方法'left'：左結合。左側のデータフレームの全ての行を残し、右側からマッチする情報を追加。
#結合方法'right'：右結合。右側のデータフレームの全ての行を残し、左側からマッチする情報を追加。
#結合方法'outer'：完全外部結合。両方のデータフレームのすべての行を結合。マッチしない部分は「NaN」になる。

In [ ]:
join_data = pd.merge(join_data, item_master, on='item_id', how='left')

In [ ]:
join_data.head()

### ノック5：必要なデータ列を作ろう

In [ ]:
join_data['price'] = join_data['quantity'] * join_data['item_price']

In [ ]:
join_data[['quantity', 'item_price', 'price']].head()

### ノック6：データ検算をしよう

In [ ]:
print(join_data['price'].sum())

In [ ]:
print(transaction['price'].sum())

### ノック7：各種統計量を把握しよう

In [ ]:
join_data.isnull().sum()

#isnull().sum()：欠損値（空白のセル）の合計。

In [ ]:
join_data.describe()

#df.describe()：基礎統計量を表示する。

In [ ]:
print(join_data['payment_date'].min())

In [ ]:
print(join_data['payment_date'].max())

### ノック8：月別でデータを集計してみよう

In [ ]:
join_data.dtypes

#df.dtypes：データフレームのクラス(データ型)を確認する。

#【df.dtypesで確認できるクラス(データ型)↓】
#  int：整数
#  float：少数
#  object：文字列（pandasが、列の種類を表示するときに使う名前。）
#  ※str：文字列（Python(言語そのもの)が呼ぶ名前。）str = object
#  datetime：日付・時間

In [ ]:
join_data['payment_date'] = pd.to_datetime(join_data['payment_date'])

#pd.to_datetime：「文字（object）」として読み込まれた日付を、計算や加工ができる『日付専用データ（datetime）』に作り変える命令。
#to：～へ変換する。

In [ ]:
join_data['payment_month'] = join_data['payment_date'].dt.strftime('%Y%m')

#dt.strftime：「dt」datetimeの略。
#strftime：時間を、形式を整えた文字にする。※strftime = string（文字）format（形式を整える）time（時間）の略。
#決まり文句：「%Y」西暦（4桁）,「%y」西暦（2桁）,「%m」月（2桁）,「%d」日（2桁）,「%H」時（2桁）,「%M」分（2桁）
#例：「%Y」2025,「%y」25,「%m」02,「%d」03,「%H」15,「%M」31

In [ ]:
join_data[['payment_date', 'payment_month']].head()

In [ ]:
join_data.groupby('payment_month')[['price']].sum()

#groupby：データを同じ仲間ごとにグループ分けする。

### ノック9：月別、商品別でデータを集計してみよう

In [ ]:
join_data.groupby(['payment_month','item_name'])[['price','quantity']].sum()

In [ ]:
pd.pivot_table(join_data, index='item_name', columns='payment_month', values=['price', 'quantity'], aggfunc='sum')

#pd.pivot_table：クロス集計表（Excelの表）を作る。※pd.pivot_table(表の名前,index=,columns=,values=,aggfunc=)
#「index=」行ラベル（縦軸）
#「columns=」列ラベル（横軸）
#「values=」値（計算対象）
#「aggfunc=」値の集計設定（基礎統計量）
#よく使う基礎統計量：sum（合計）,count（個数）,mean（平均値）,std（標準偏差）,median（中央値）

### ノック10：商品別の売上推移を可視化してみよう

In [ ]:
graph_data = pd.pivot_table(join_data, index='payment_month', columns='item_name', values='price',aggfunc='sum')

In [ ]:
graph_data.head()

In [ ]:
import matplotlib.pyplot as plt

#matplotlib：全ての工程を自らの手で制御する「プロ仕様版」グラフ作成ツール。
#matplotlib.pyplot：複雑な工程を自動化した「簡易操作版」グラフ作成ツール。

In [ ]:
%matplotlib inline

#マジックコマンド（% , %%）：プログラムの外側の設定を変更する。
# %（ラインマジック）：その一行だけに効く命令。
# %%（セルマジック）：そのセル全体に効く命令。

#【よく使うマジックコマンド（↓）】
#%matplotlib inline（グラフをノートブック内に直接表示する。）
#%timeit（その行のコードの実行速度を計測する。）
#%pwd（現在のフォルダにあるファイル一覧を表示する。）
#%run（外部のPythonファイルをノートブック上で実行する。）

In [ ]:
plt.plot(list(graph_data.index), graph_data['PC-A'], label='PC-A')
plt.legend()

#plt.plot（折れ線グラフ）
#plt.bar（棒グラフ）
#plt.scatter（散布図）
#plt.hist（ヒストグラム）

#list(graph_data.index)：x軸(横軸)の値。
#list()：行ラベルをグラフのx軸(横軸)に使うために「データの列」に変更する。
#index：行ラベル。
#graph_data['PC-A']：縦軸（y軸）の値。列名「PC-A」のデータ。
#label='PC-A'：凡例ラベル。

#plt.legend()：グラフに凡例ラベルを表示させる。

In [ ]:
plt.plot(list(graph_data.index), graph_data['PC-B'], label='PC-B')
plt.legend()

In [ ]:
plt.plot(list(graph_data.index), graph_data['PC-C'], label='PC-C')
plt.legend()

In [ ]:
plt.plot(list(graph_data.index), graph_data['PC-D'], label='PC-D')
plt.legend()

In [ ]:
plt.plot(list(graph_data.index), graph_data['PC-E'], label='PC-E')
plt.legend()